Banking loan & risk analysis

Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Upload datasets

In [2]:
customers = pd.read_csv("/content/Bank loan Risk Analytics/customers.csv")
loans = pd.read_csv("/content/Bank loan Risk Analytics/loans.csv")
payments = pd.read_csv("/content/Bank loan Risk Analytics/loan_payments.csv")

Merge Customer and Loan Data

In [3]:
df = loans.merge(customers, on="customer_id", how="left")

Data Quality Checking

In [4]:
print(df.info())
print(df.isna().sum())
print(df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   loan_id          800 non-null    int64  
 1   customer_id      800 non-null    int64  
 2   branch_id        800 non-null    int64  
 3   loan_type        800 non-null    object 
 4   purpose          800 non-null    object 
 5   loan_amount      800 non-null    int64  
 6   tenure_months    800 non-null    int64  
 7   interest_rate    800 non-null    float64
 8   dti              800 non-null    float64
 9   loan_status      800 non-null    object 
 10  approval_status  800 non-null    object 
 11  customer_name    800 non-null    object 
 12  age              800 non-null    int64  
 13  annual_income    800 non-null    int64  
 14  city             800 non-null    object 
 15  occupation       800 non-null    object 
 16  credit_score     800 non-null    int64  
 17  existing_loans  

Risk features

In [5]:
df["credit_band"] = pd.cut(
    df["credit_score"],
    bins=[0,599,679,749,900],
    labels=["Poor","Fair","Good","Excellent"]
)
df["dti_band"] = pd.cut(
    df["dti"],
    bins=[-np.inf,0.35,0.50,np.inf],
    labels=["Low","Medium","High"]
)
df["default_flag"] = (df["loan_status"] == "Defaulted").astype(int)
df["high_risk_flag"] = (
    (df["credit_score"] < 600) |
    (df["dti"] > 0.50) |
    (df["default_flag"] == 1)
).astype(int)

KPI calculations

In [6]:
approved = df[df["approval_status"]=="Approved"]
print("Approved loan value:", approved["loan_amount"].sum())
print("Default rate:", approved["default_flag"].mean())
print("Average DTI:", approved["dti"].mean())

Approved loan value: 1036240000
Default rate: 0.22122762148337596
Average DTI: 5.148511508951406


Segment analysis

In [7]:
risk_by_credit = df.groupby("credit_band", observed=True).agg(
    loans=("loan_id","count"),
    default_rate=("default_flag","mean"),
    avg_loan=("loan_amount","mean")
).reset_index()
print(risk_by_credit)

risk_by_type = df.groupby("loan_type").agg(
    loans=("loan_id","count"),
    portfolio_value=("loan_amount","sum"),
    default_rate=("default_flag","mean")
).reset_index().sort_values("portfolio_value", ascending=False)
print(risk_by_type)

  credit_band  loans  default_rate      avg_loan
0        Poor     87      0.287356  1.268448e+06
1        Fair    295      0.261017  1.312525e+06
2        Good    266      0.154135  1.344342e+06
3   Excellent    152      0.197368  1.343717e+06
        loan_type  loans  portfolio_value  default_rate
0       Auto Loan    173        233965000      0.231214
1   Business Loan    174        226070000      0.212644
2  Education Loan    156        211100000      0.243590
3       Home Loan    151        204150000      0.198675
4   Personal Loan    146        184105000      0.191781


Payment analysis

In [8]:
payment_summary = payments.groupby("loan_id").agg(
    payments=("payment_id","count"),
    avg_days_late=("days_late","mean"),
    late_payments=("payment_status", lambda x: (x=="Late").sum())
).reset_index()
payment_summary["late_rate"] = payment_summary["late_payments"]/payment_summary["payments"]

df = df.merge(payment_summary, on="loan_id", how="left")

Export Power BI-ready dataset

In [9]:
df.to_csv("powerbi_loan_risk.csv", index=False)
payments.to_csv("powerbi_payments.csv", index=False)

Chart Plotting

In [10]:
risk_by_credit.plot(x="credit_band", y="default_rate", kind="bar", legend=False)
plt.title("Default Rate by Credit Band")
plt.ylabel("Default Rate")
plt.tight_layout()
plt.savefig("default_rate_by_credit_band.png", dpi=200)
plt.close()
